# M??teri Kay?p (Churn) Analizi-1

Yararlanilan kaynak: 
 - Deniz Kilinc, Makine ??renimi ve Derin ??renme ile M??teri Kay?p (Churn) Analizi-1
 - https://medium.com/deep-learning-turkiye/makine-??renimi-ve-derin-??renme-ile-m??teri-kay?p-churn-analizi-1-63a4513b8a6f

In [ ]:
import numpy as np
#sonu?lar?n yeniden ?retilebilir olmas?n? ama?l?yoruz
np.random.seed(1)
from sklearn import cross_validation
from sklearn import preprocessing
import pandas as pd
columns = [
     'state',
     'account length', 
     'area code', 
     'phone number', 
     'international plan', 
     'voice mail plan', 
     'number vmail messages',
     'total day minutes',
     'total day calls',
     'total day charge',
     'total eve minutes',
     'total eve calls',
     'total eve charge',
     'total night minutes',
     'total night calls',
     'total night charge',
     'total intl minutes',
     'total intl calls',
     'total intl charge',
     'number customer service calls',
     'churn']
data = pd.read_csv('churndataset.txt', header = None, names = columns)
#Datasetin orjinali hali
print("Dataset orjinal hali: " + str(data.shape))

In [ ]:
data.head(7)

# ?n ??leme

In [ ]:
import re
s = 'string. '
s = re.sub(r'[^\w\s]','',s)
s

In [ ]:
import re

def map2binary(x): 
    x = re.sub(r'[^\w\s]','',str(x))
    x = x.strip().lower()
    if x == 'false': return 0
    if x == 'true': return 1 
    if x == 'yes': return 0
    if x == 'no': return 1   
 

In [ ]:
data['churn'] = data['churn'].apply(map2binary)
data['voice mail plan'] = data['voice mail plan'].apply(map2binary)
data['international plan'] = data['international plan'].apply(map2binary)

data[['churn','voice mail plan','international plan']].head(10)

In [ ]:
#Preprocessing Ad?m 2: phone number, area code, state ?zniteliklerinin kald?r?lmas?
data.drop('phone number', axis = 1, inplace = True)
data.drop('area code', axis = 1, inplace = True)
data.drop('state', axis = 1, inplace = True)
print("Dataset preprocessing sonrasi: " + str(data.shape))

In [ ]:
print("Veri turleri:")
print(data.dtypes)

## S?n?f dengesizli?i (class imbalance) problemi

In [ ]:
data1 = data[data['churn']==1]
print("Churn olanlar-data1:"+ str(data1.shape))
data2 = data[data['churn']==0]
print("Churn olmayanlar-data2:"+ str(data2.shape))

In [ ]:
#Her iki sinifta da esit sayida ornek olmasini istiyoruz
data = data1.append(data2[:483])
print("Son veriseti :"+ str(data.shape))

# Siniflandirma
6 tane temel (base) ve 3 tane kolektif (ensemble) 

 - Temel S?n?fland?rma Modelleri: 
    > Logistic Regression, Naive Bayes, Decision Tree (CART), K-NN, SVM(Support Vector Machines), LDA (Linear Discriminant Analysis)
 - Kolektif S?n?fland?rma Modelleri: 
    > AdaBoostClassifier, BaggingClassifier, RandomForestClassifier

In [ ]:
X = data.iloc[:, data.columns != 'churn']
y = data.churn

# Splitting the dataset into the Training set and Test set
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state = 0)

# veri setimizi -1 ve 1 aras?nda ?l?eklendiriyoruz
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler((-1,1))
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# S?n?fland?rma Modellerine Ait K?t?phaneler
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, BaggingClassifier

models = []
models.append(('Logistic Regression', LogisticRegression()))
models.append(('Naive Bayes', GaussianNB()))
models.append(('Decision Tree (CART)',DecisionTreeClassifier())) 
models.append(('K-NN', KNeighborsClassifier()))
models.append(('SVM', SVC()))
models.append(('LDA', LinearDiscriminantAnalysis()))
models.append(('AdaBoostClassifier', AdaBoostClassifier()))
models.append(('BaggingClassifier', BaggingClassifier()))
models.append(('RandomForestClassifier', RandomForestClassifier()))

# Modelleri test edelim
for name, model in models:
    model = model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    from sklearn import metrics
    print("Model -> %s -> ACC: %%%.2f" % (name,metrics.accuracy_score(y_test, y_pred)*100))

In [ ]:
import matplotlib.pyplot as plt
num_bins = 10
data.hist(bins=num_bins, figsize=(20,15))
plt.savefig("churn_histogram")
plt.show()

# Deep learning ve Neural Networks

Burada `X_train` degil `X_train_scaled` kullanmayi unutmayin!!

Uzakliga bagli makine ogrenme algoritmalari, fetaure'larin normalize edilmesiin gerektirir.


In [ ]:
from keras.models import Sequential
from keras.layers.core import Dense, Dropout, Activation, Flatten
from keras.layers.noise import GaussianNoise
from keras.layers.normalization import BatchNormalization
from keras.layers.advanced_activations import PReLU
from keras.utils import np_utils

In [ ]:
# 100 epochs, gradient batched each 100 samples
batch_size = 100
nb_epoch = 1000

In [ ]:
model = Sequential()

model.add(Dense(50, input_shape=(17,)))
model.add(Activation("hard_sigmoid"))
model.add(BatchNormalization())
model.add(Dropout(0.1))

model.add(Dense(10))
model.add(Activation("hard_sigmoid"))
model.add(BatchNormalization())
model.add(Dropout(0.1))

model.add(Dense(1))
model.add(Activation('sigmoid'))

model.compile(loss='binary_crossentropy', optimizer='adam')

model.fit(X_train_scaled, y_train, batch_size=batch_size, epochs=nb_epoch, verbose=0)

# Predicting the Test set results
y_pred = model.predict(X_test_scaled)
y_pred = (y_pred > 0.5) * 1

# Making the Confusion Matrix
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test, y_pred)
print("confusion_matrix:\n", cm)
print("accuracy_score: ", accuracy_score(y_test, y_pred))

In [ ]:
from sklearn.neural_network import MLPClassifier
nn=MLPClassifier(hidden_layer_sizes=(7,7), max_iter=1000, alpha=0.01,
                     solver='sgd', verbose=0,  random_state=21,tol=0.000000001)

nn.fit(X_train_scaled,y_train)
# Predicting the Test set results
y_pred = nn.predict(X_test_scaled)


# Making the Confusion Matrix
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test, y_pred)
print("confusion_matrix:\n", cm)
print("accuracy_score: ", accuracy_score(y_test, y_pred))